# ДЗ №1. ETL: население стран из API Всемирного банка

## Описание

Получим показатель `SP.POP.TOTL` для десяти стран за 2000–2024 годы и загрузим его в `stg.population` в PostgreSQL. Источник — [World Bank Indicators API v2](https://api.worldbank.org/v2/), целевая БД запускается через Docker Compose.

При настройках из `.env.example` выборка содержит до 250 наблюдений, разбитых на пять страниц. Строка таблицы соответствует одной стране и одному году. Пропущенное население не заменяем нулём: такие наблюдения не загружаем.

Для запуска с нуля выполните `bash run.sh` из корня проекта. Скрипт подготовит окружение и `.env`, поднимет PostgreSQL и выполнит все ячейки. После этого ноутбук можно открыть в Jupyter и повторно выполнить **Restart Kernel and Run All Cells** без правок кода.

## Импорты

Функции ETL вынесены в `etl.py`, чтобы отдельно проверить пагинацию и преобразования. Здесь последовательно запускаются все три этапа и SQL-проверки.

In [1]:
import logging
import sys
from contextlib import closing
from pathlib import Path

import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv
from IPython.display import display

## Конфигурация

Параметры подключения и выборки берём из `.env`. Пароль не выводим. Лог сохраняется в `logs/etl.log`; эта папка исключена из Git.

In [2]:
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in candidates if (path / "etl.py").is_file() and (path / "init/init.sql").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Запустите ноутбук из hw-1 или hw-1/notebooks.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=False)

from etl import (
    TARGET_COLUMNS,
    create_http_session,
    connect_to_postgres,
    extract_population,
    load_population,
    read_config,
    transform_population,
)

config = read_config(project_root)
(project_root / "logs").mkdir(exist_ok=True)
logger = logging.getLogger("population_etl")
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for handler in [
    logging.StreamHandler(sys.stdout),
    logging.FileHandler(project_root / "logs/etl.log", encoding="utf-8"),
]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

display(pd.DataFrame({
    "Параметр": ["Страны", "Годы", "Размер страницы", "Целевая таблица"],
    "Значение": [", ".join(config.countries), f"{config.start_year}–{config.end_year}", config.page_size, "stg.population"],
}))

,Параметр,Значение
0,Страны,"BRA, CHN, IND, ZAF, RUS, USA, DEU, FRA, JPN, GBR"
1,Годы,2000–2024
2,Размер страницы,50
3,Целевая таблица,stg.population


## Extract

У API ответ состоит из метаданных и списка записей. Прочитаем все страницы и сверим количество полученных строк с `total`. Если между запросами поменяется версия источника или запрос завершится ошибкой, загрузка в БД не начнётся. Для временных сетевых ошибок настроены таймаут и повторы.

In [3]:
with create_http_session() as session:
    raw_records, source_updated_at = extract_population(config, session)

display(pd.DataFrame({
    "Строк извлечено": [len(raw_records)],
    "Дата обновления источника": [source_updated_at],
}))
display(pd.json_normalize(raw_records).head())

2026-11-24 21:05:24,945 | INFO | Extract: страница 1/5, строк 50.


2026-11-24 21:05:24,981 | INFO | Extract: страница 2/5, строк 50.


2026-11-24 21:05:25,020 | INFO | Extract: страница 3/5, строк 50.


2026-11-24 21:05:25,061 | INFO | Extract: страница 4/5, строк 50.


2026-11-24 21:05:25,096 | INFO | Extract: страница 5/5, строк 50.


2026-11-24 21:05:25,098 | INFO | Extract завершён: 250 строк, версия источника 2026-11-11.


,Строк извлечено,Дата обновления источника
0,250,2026-11-11


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,BRA,2024,211998573,,,0,SP.POP.TOTL,"Population, total",BR,Brazil
1,BRA,2023,211140729,,,0,SP.POP.TOTL,"Population, total",BR,Brazil
2,BRA,2022,210306415,,,0,SP.POP.TOTL,"Population, total",BR,Brazil
3,BRA,2021,209550294,,,0,SP.POP.TOTL,"Population, total",BR,Brazil
4,BRA,2020,208660842,,,0,SP.POP.TOTL,"Population, total",BR,Brazil


## Transform

Распакуем вложенные поля, переименуем колонки и приведём год и население к целым числам. Удалим пропуски населения и точные дубли. Отрицательные и дробные значения населения, чужие коды стран и противоречивые дубли считаем ошибкой данных, поэтому не исправляем их догадками.

Дата обновления источника сохраняется отдельно от времени записи строки в нашу БД.

In [4]:
population = transform_population(raw_records, source_updated_at, config)

display(population.head())
display(population.dtypes.rename("Тип в pandas").to_frame())
display(pd.DataFrame({
    "Строк до преобразования": [len(raw_records)],
    "Строк к загрузке": [len(population)],
    "Стран": [population["country_code"].nunique()],
    "Первый год": [population["year"].min()],
    "Последний год": [population["year"].max()],
}))

2026-11-24 21:05:25,130 | INFO | Transform: 250 -> 250 строк; пропусков населения 0, точных дублей 0.


,country_code,country_name,year,population,indicator_code,source_updated_at
0,BRA,Brazil,2000,174018282,SP.POP.TOTL,2026-11-11
1,BRA,Brazil,2001,176301203,SP.POP.TOTL,2026-11-11
2,BRA,Brazil,2002,178503484,SP.POP.TOTL,2026-11-11
3,BRA,Brazil,2003,180622688,SP.POP.TOTL,2026-11-11
4,BRA,Brazil,2004,182675143,SP.POP.TOTL,2026-11-11


,Тип в pandas
country_code,string[python]
country_name,string[python]
year,int64
population,int64
indicator_code,object
source_updated_at,object


,Строк до преобразования,Строк к загрузке,Стран,Первый год,Последний год
0,250,250,10,2000,2024


## Load

Запишем данные одной транзакцией. Первичный ключ — `(country_code, year)`. `ON CONFLICT DO UPDATE` обновит пересмотренные значения Всемирного банка, а одинаковые строки оставит без изменений. При ошибке соединения или SQL появится понятное сообщение; частичная загрузка не сохраняется.

In [5]:
processed_rows = load_population(population, config)
display(pd.DataFrame({"Таблица": ["stg.population"], "Обработано строк": [processed_rows]}))

2026-11-24 21:05:25,259 | INFO | Load завершён: обработано 250 строк в stg.population.


,Таблица,Обработано строк
0,stg.population,250


## Проверка

Выполним `SELECT COUNT(*)`, выведем первые пять строк и проверим, что каждая подготовленная запись есть в БД. Затем повторно загрузим тот же пакет и сравним полное содержимое таблицы, включая `loaded_at`. Так проверяется и отсутствие дублей, и отсутствие изменений при повторе.

In [6]:
def read_database_state():
    with closing(connect_to_postgres(config)) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT COUNT(*) FROM stg.population")
            row_count = cursor.fetchone()[0]
            cursor.execute("""
                SELECT country_code, country_name, year, population,
                       indicator_code, source_updated_at, loaded_at
                FROM stg.population
                ORDER BY country_code, year
            """)
            all_rows = cursor.fetchall()
            cursor.execute("""
                SELECT country_code, country_name, year, population,
                       indicator_code, source_updated_at, loaded_at
                FROM stg.population
                ORDER BY country_code, year
                LIMIT 5
            """)
            first_five = cursor.fetchall()
            cursor.execute("""
                SELECT COUNT(*) FROM (
                    SELECT country_code, year
                    FROM stg.population
                    GROUP BY country_code, year
                    HAVING COUNT(*) > 1
                ) AS duplicate_keys
            """)
            duplicate_groups = cursor.fetchone()[0]
    return row_count, all_rows, first_five, duplicate_groups


count_before, rows_before, first_five, duplicate_groups = read_database_state()
expected_rows = set(population[TARGET_COLUMNS].itertuples(index=False, name=None))
actual_rows = {tuple(row[:len(TARGET_COLUMNS)]) for row in rows_before}
assert expected_rows.issubset(actual_rows), "Не все подготовленные данные найдены в БД."
assert duplicate_groups == 0, "В таблице есть повторяющиеся ключи."
assert count_before == len(rows_before), "Количество строк изменилось во время проверки."

load_population(population, config)
count_after, rows_after, _, duplicate_groups_after = read_database_state()
assert count_before == count_after, "Повторная загрузка изменила количество строк."
assert rows_before == rows_after, "Повторная загрузка изменила содержимое таблицы."
assert duplicate_groups_after == 0, "После повторной загрузки появились дубли."

checks = pd.DataFrame({
    "Проверка": [
        "Извлечено из API", "Подготовлено к загрузке", "SELECT COUNT(*) до повтора",
        "SELECT COUNT(*) после повтора", "Групп с повторяющимся ключом",
        "Все подготовленные строки есть в БД", "Повторная загрузка сохранила содержимое",
    ],
    "Результат": [
        len(raw_records), len(population), count_before, count_after,
        duplicate_groups_after, True, rows_before == rows_after,
    ],
})
display(checks)
display(pd.DataFrame(first_five, columns=[*TARGET_COLUMNS, "loaded_at"]))

2026-11-24 21:05:25,396 | INFO | Load завершён: обработано 250 строк в stg.population.


,Проверка,Результат
0,Извлечено из API,250
1,Подготовлено к загрузке,250
2,SELECT COUNT(*) до повтора,250
3,SELECT COUNT(*) после повтора,250
4,Групп с повторяющимся ключом,0
5,Все подготовленные строки есть в БД,True
6,Повторная загрузка сохранила содержимое,True


,country_code,country_name,year,population,indicator_code,source_updated_at,loaded_at
0,BRA,Brazil,2000,174018282,SP.POP.TOTL,2026-11-11,2026-11-24 21:04:41.677319+00:00
1,BRA,Brazil,2001,176301203,SP.POP.TOTL,2026-11-11,2026-11-24 21:04:41.677319+00:00
2,BRA,Brazil,2002,178503484,SP.POP.TOTL,2026-11-11,2026-11-24 21:04:41.677319+00:00
3,BRA,Brazil,2003,180622688,SP.POP.TOTL,2026-11-11,2026-11-24 21:04:41.677319+00:00
4,BRA,Brazil,2004,182675143,SP.POP.TOTL,2026-11-11,2026-11-24 21:04:41.677319+00:00


## Вывод

В PostgreSQL загружено **250 наблюдений**: население десяти стран за 2000–2024 годы. В этой выборке не оказалось пропусков населения и точных дублей, поэтому после преобразований сохранились все строки. Проверка SQL подтвердила, что каждая подготовленная запись есть в таблице.

Повторная загрузка оставила **250 строк** и не изменила их содержимое, включая `loaded_at`. Значит, одинаковый пакет можно загружать повторно без дублей и лишних обновлений. Если Всемирный банк пересмотрит данные, следующий запуск обновит соответствующие наблюдения.

Дата версии источника — **2026-11-11**. Она относится ко всему набору данных; время записи в нашу БД хранится отдельно. Строки, которые источник перестал отдавать, текущий UPSERT не удаляет — для полного совпадения со снимком API потребуется отдельная обработка удалений.